In this notebook we implement pybdshadow for calculating building's shadowing in bologna's city for adressing a Urban Thermal Adaptability index 

In [ ]:
!pip install pybdshadow

In [ ]:
import pandas as pd
import geopandas as gpd
import pybdshadow
import matplotlib.pyplot as plt
import time
import os

import warnings
warnings.filterwarnings("ignore", message="CRS not set for some of the concatenation inputs")


In [ ]:
# user-defined paths
base_dir = '../../../../datos-notebooks/2.global-data-bologna/'
buildings_file_name = 'GBA_bologna.geojson'
date = '2024-08-09 09:58:10'

In [ ]:
output_folder = os.path.join(base_dir, 'MRT-prediction/shadows/')

In [ ]:
#Read building data
buildings = gpd.read_file(os.path.join(base_dir, 'objects-height',buildings_file_name))
buildings.head()

In [ ]:
buildings.crs

In [ ]:
# preprocess the data
buildings = pybdshadow.bd_preprocess(buildings)

In [ ]:
# If the date we are using is Local time and not UTC, run this:
'''
#Given UTC time
date = pd.to_datetime('2024-06-07 08:41:01')\
    .tz_localize('Europe/Rome') \
    .tz_convert('UTC')
''' 

In [ ]:
# date and time in UTC
date = pd.to_datetime(date).tz_localize('UTC')

In [ ]:
# Verify if it's correct
print(date)

In [ ]:
buildings.geom_type.value_counts()

In [ ]:
# compute shadows
start = time.time()

shadows = pybdshadow.bdshadow_sunlight(buildings, date, roof=False, include_building=False) #if roof=true, the output also has the shadows proyected on roofs

print(f"Proccesing time: {round(time.time() - start, 2)} seconds")
shadows

In [ ]:
# Visualize buildings and shadows using matplotlib
fig = plt.figure(1, (5, 5))
ax = plt.subplot(111)

# plot buildings
buildings.plot(ax=ax)

# plot shadows
shadows.plot(ax=ax, alpha=0.7,
             column='type',
             categorical=True,
             cmap='Set1_r',
             legend=True)

plt.show()


In [ ]:
print(shadows.crs)  
print(shadows.shape[0]) 

In [ ]:
os.makedirs(output_folder, exist_ok=True)
date_str = date.replace(' ', '_').replace(':', '')
shadows.to_file(os.path.join(output_folder, f"shadows_buildings_{date_str}.gpkg"), driver="GPKG")